# 📊 Análise Exploratória de Dados (EDA) — Telco Customer Churn

**Objetivo:** Compreender os fatores que influenciam o cancelamento de clientes (churn) em uma empresa de telecomunicações, identificar anomalias, avaliar o desbalanceamento de classes e garantir que nenhuma informação futura (*data leakage*) contamine nosso pipeline de Machine Learning.

### Sumário do Estudo:
1. **Visão Geral & Tipagem de Dados**
2. **Diagnóstico de Valores Ausentes & Anomalias**
3. **Distribuição do Alvo (`Churn`) & Desbalanceamento**
4. **Análise das Variáveis Numéricas & Detecção de Outliers**
5. **Análise de Variáveis Categóricas & Cardinalidade**
6. **Correlação Linear & Insights de Negócio**
7. **Mapeamento de Riscos de Vazamento de Dados (*Data Leakage*)**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11

# Carregamento dos dados brutos
df_raw = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print(f'Dimensões: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas')
df_raw.head(3)

## 1. Diagnóstico de Valores Ausentes & Correção de `TotalCharges`

No dataset original, a coluna `TotalCharges` aparece como tipo `object` (texto) em vez de numérico.
Identificamos que existem 11 clientes cujo valor está preenchido com espaços em branco (`' '`).

> **Explicação de Negócio:** Todos os 11 clientes possuem tempo de casa (`tenure`) igual a **0**, ou seja, acabaram de assinar o plano e ainda não completaram o primeiro ciclo de faturamento. Portanto, o valor correto acumulado é exatamente **0.0**.

In [ ]:
# Verificando os espaços em branco
blanks = (df_raw['TotalCharges'].astype(str).str.strip() == '').sum()
print(f'Registros com TotalCharges em branco: {blanks}')

# Conversão robusta para float e preenchimento com 0.0
df = df_raw.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
print(f'Valores nulos após o tratamento: {df["TotalCharges"].isnull().sum()}')

## 2. Distribuição do Alvo (`Churn`) e Grau de Desbalanceamento

- **Não-Churn (`No`):** ~73.4% dos clientes
- **Churn (`Yes`):** ~26.6% dos clientes
- **Razão de Desbalanceamento:** 2.77 para 1

> ⚠️ **Implicação de Modelagem:** Este desbalanceamento moderado significa que um classificador ingênuo que sempre prever 'Não' teria 73.4% de acurácia, mas seria inútil para o negócio. Portanto:
> 1. Acurácia **não** será utilizada como métrica principal.
> 2. Priorizaremos **ROC-AUC**, **PR-AUC**, **Recall** (taxa de detecção de churn) e **F1-Score**.
> 3. Testaremos técnicas de ajuste de peso como `scale_pos_weight` no XGBoost e calibração de threshold financeiro.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Churn', data=df, hue='Churn', palette=['#3498db', '#e74c3c'], legend=False, ax=ax)
ax.set_title('Distribuição da Variável Alvo (Churn)', fontweight='bold')
ax.set_xlabel('Cancelou o Serviço?')
ax.set_ylabel('Quantidade')
for p in ax.patches:
    h = p.get_height()
    pct = (h / len(df)) * 100
    ax.annotate(f'{h}\n({pct:.1f}%)', (p.get_x() + p.get_width() / 2., h / 2), ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Variáveis Numéricas e Avaliação de Outliers

As variáveis contínuas são:
- `tenure`: Número de meses em que o cliente permaneceu na empresa (0 a 72 meses).
- `MonthlyCharges`: Valor mensal cobrado do cliente ($ 18.25 a $ 118.75).
- `TotalCharges`: Valor total acumulado cobrado ($ 0 a $ 8,684.80).

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    sns.kdeplot(data=df, x=col, hue='Churn', common_norm=False, fill=True, palette=['#3498db', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'Distribuição de {col}', fontweight='bold')
plt.tight_layout()
plt.show()

# Verificação de outliers com método IQR (Tukey)
stats = df[num_cols].describe().T
stats['IQR'] = stats['75%'] - stats['25%']
stats['Lower'] = stats['25%'] - 1.5 * stats['IQR']
stats['Upper'] = stats['75%'] + 1.5 * stats['IQR']
stats['Outliers'] = [((df[c] < stats.loc[c, 'Lower']) | (df[c] > stats.loc[c, 'Upper'])).sum() for c in num_cols]
stats[['min', 'mean', '50%', 'max', 'IQR', 'Outliers']]

> **Principais Insights Numéricos:**
- **`tenure`:** Clientes que cancelam concentram-se fortemente nos primeiros **1 a 12 meses** de contrato. Clientes veteranos (> 60 meses) raramente dão churn.
- **`MonthlyCharges`:** Clientes com mensalidades mais altas ($ 70 - $ 100) têm probabilidade significativamente maior de cancelamento.
- **Outliers:** Nenhuma variável contínua possui outliers artificiais ou pontos extremos anômalos. O método de escala `StandardScaler` é ideal e seguro.

## 4. Variáveis Categóricas e Análise de Cardinalidade

Vamos inspecionar a quantidade de valores distintos por coluna categórica para determinar a melhor estratégia de encoding.

In [ ]:
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]
cardinality = {col: {'Valores Únicos': df[col].nunique(), 'Exemplos': list(df[col].unique())} for col in cat_cols}
pd.DataFrame(cardinality).T

> **Diagnóstico de Cardinalidade:**
- Todas as colunas possuem entre 2 e 4 categorias distintas.
- **Não há alta cardinalidade**, portanto o `OneHotEncoder(handle_unknown='ignore')` é a estratégia recomendada: cria apenas ~45 colunas no total, sem risco de explosão dimensional ou necessidade de técnicas complexas como Target Encoding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Contract', hue='Churn', palette=['#3498db', '#e74c3c'], ax=axes[0])
axes[0].set_title('Churn por Tipo de Contrato', fontweight='bold')

sns.countplot(data=df, x='PaymentMethod', hue='Churn', palette=['#3498db', '#e74c3c'], ax=axes[1])
axes[1].set_title('Churn por Forma de Pagamento', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

> **Insights de Negócio:**
1. **Tipo de Contrato (`Contract`):** Contratos mensais (`Month-to-month`) respondem por quase todo o volume de churn. Contratos de 1 ou 2 anos geram retenção altíssima.
2. **Forma de Pagamento (`PaymentMethod`):** Clientes que pagam com `Electronic check` apresentam taxa de cancelamento substancialmente mais alta do que os que usam débito automático em conta ou cartão de crédito.

## 5. Mapeamento de Fontes de Vazamento de Dados (*Data Leakage*)

O vazamento de dados ocorre quando o modelo de Machine Learning tem acesso em tempo de treino a informações que não estariam disponíveis no momento da predição real em produção.

| Coluna / Processo | Diagnóstico de Risco | Mitigação Obrigatória |
| :--- | :--- | :--- |
| `customerID` | **Alto:** Identificador arbitrário. O modelo pode memorizar IDs específicos gerando overfitting severo sem valor preditivo. | **Eliminar imediatamente** antes do pré-processamento. |
| `TotalCharges` | **Médio:** Em alguns sistemas, taxas de quebra de contrato pós-cancelamento são somadas à conta final. No Telco Churn, a coluna é estritamente $\approx \text{tenure} \times \text{MonthlyCharges}$. | **Segura para uso**, mantida como feature. |
| **Pré-processamento Global** | **Crítico:** Ajustar transformadores (`StandardScaler.fit()`, `OneHotEncoder.fit()`) sobre o conjunto completo vaza a média, desvio e vocabulário do teste para o treino. | **Encapsular tudo via `Pipeline` e `ColumnTransformer`**, chamando `.fit()` exclusivamente em `X_train`. |